# Gradient Boosting & XGBoost
Accompanies **Chapter 16: Gradient Boosting Machines** and **Chapter 17: Extreme Gradient Boosting (XGBoost)**.

Gradient boosting builds ensembles sequentially by fitting base learners to pseudo-residuals (negative loss gradients):
$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \sum_{j=1}^J w_{jm} \mathbb{I}(\mathbf{x} \in R_{jm})$$

XGBoost expands the objective using a second-order Taylor expansion around the current prediction:
$$\mathcal{L}^{(t)} \approx \sum_{i=1}^n \left[ g_i f_t(\mathbf{x}_i) + \frac{1}{2} h_i f_t^2(\mathbf{x}_i) \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2$$
where $g_i = \partial_{\hat{y}^{(t-1)}} \ell$ and $h_i = \partial^2_{\hat{y}^{(t-1)}} \ell$.

Grouping instances by leaf $j$ ($G_j = \sum_{i \in I_j} g_i$, $H_j = \sum_{i \in I_j} h_i$) yields the closed-form optimal leaf weights and split gain:
$$w_j^* = -\frac{G_j}{H_j + \lambda}, \quad \text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X = df[numeric_cols].fillna(df[numeric_cols].median())
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Tracking Sequential Staged Deviance
We evaluate loss after each boosting iteration to monitor how learning rate shrinkage ($\eta$) controls sequential overfitting.

In [ ]:
gbm = GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

train_loss = [log_loss(y_train, p) for p in gbm.staged_predict_proba(X_train)]
test_loss = [log_loss(y_test, p) for p in gbm.staged_predict_proba(X_test)]

plt.figure(figsize=(9, 4.8))
plt.plot(train_loss, label='Train Log-Loss', color='royalblue', linewidth=2)
plt.plot(test_loss, label='Test Log-Loss', color='crimson', linewidth=2)
plt.xlabel('Boosting Round ($t$)', fontsize=11)
plt.ylabel('Negative Log-Likelihood (Deviance)', fontsize=11)
plt.title('Sequential Loss Reduction & Early Generalization Gap', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

best_round = np.argmin(test_loss)
print(f"Optimal test loss at round {best_round + 1}: {test_loss[best_round]:.4f}")